# Flood exposure and EJ designation in Delaware County, PA
Thomas P. McKeon, MPH 

PhD Candidate GUS
Temple University

This notebook runs the analysis behind *Indicator or Domain? Flood Exposure and Environmental Justice Designation along the Delaware River* and prints the numbers reported in the Results section.

**What it does, in order**
1. Reads PennEnviroScreen (PES) v1.1 for every block group in Pennsylvania.
2. Rebuilds the PES scores from the published indicators and checks them against DEP's values.
3. Measures Community Flood Exposure (CFE): the share of each block group's land under 0.5, 1.0 and 3.0 ft of water in FEMA's 1% annual-chance flood.
4. Builds the five versions of the index compared in the paper:
   - **Model 0**: official PES (ranked against the whole state)
   - **Model 1**: PES re-ranked within Delaware County
   - **Model 2**: Model 1 with CFE in place of the PES Flood Risk indicator
   - **Model 3**: Model 1 with CFE added as its own domain next to Pollution Burden
   - **Model 3A**: Pollution Burden × Population Characteristics × CFE (for comparison only)
5. Compares which block groups gain or lose EJ designation between models.
6. Repeats the comparisons under the sensitivity tests in Section 3.6.
7. Repeats the main comparisons with PES v1.0 (2010 block groups).

**Needs:** pandas, numpy, geopandas. Rebuilding CFE from the FEMA depth grids also needs rasterio, but if the saved flood-share file (`CFE_CACHE`) exists the grids are not read at all.

**Note on names:** earlier versions of this work called the flood measure "CFV". The saved flood-share files still use `CFV_` in their column names; they are renamed to `CFE_` when read.

Run the cells from top to bottom. Update path names as needed.


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd


## File locations (change these if the files move)

In [2]:
PROJECT = Path(r"C:\Users\ureka\OneDrive\Documents\ArcGIS\Projects\ClimateEJ_June22")
RESULTS = Path(r"C:\Users\ureka\OneDrive\Desktop\CFV-Model-Results")

PES_FILES = {
    "v1.1": PROJECT / "Environmental_Justice_Areas_-_PennEnviroScreen_v1.1.geojson",
    "v1.0": PROJECT / "Environmental_Justice_Areas_-_PennEnviroScreen_v1.0.geojson",
}
STATE_ROWS = {"v1.1": 10172, "v1.0": 9739}      # block groups expected in each statewide file

# Flood shares saved from an earlier run. If a file is missing it is rebuilt from the depth grids.
CFE_CACHE = {
    "v1.1": RESULTS / "revised_v1.1" / "CFV_corrected_raw.csv",
    "v1.0": RESULTS / "revised" / "CFV_corrected_raw.csv",
}

# 2020 Census population (Table P1) for Delaware County block groups
POPULATION_FILE = PROJECT / "DECENNIALPL2020.P1_2026-10-10T134340" / "DECENNIALPL2020.P1-Data.csv"

# FEMA Flood Risk Database depth grids for the 1% annual-chance flood
LOWER_DELAWARE = PROJECT / "FRD_02040202_PA_GeoTIFFs_20160801"
BRANDYWINE = PROJECT / "FRD_02040205_Geotiffs_20170801" / "FRD_02040205_Geotiffs"
DEPTH_GRIDS = {
    "riverine_lower_delaware": LOWER_DELAWARE / "Depth_01pct.tif",
    "riverine_brandywine": BRANDYWINE / "Depth_01pct.tif",
    "coastal_lower_delaware": LOWER_DELAWARE / "CstDpth01pct.tif",
}
WATER_FILE = PROJECT / "tl_2020_42045_areawater" / "tl_2020_42045_areawater.shp"   # Census open water

RUN_VERSION_1_0 = True       # set to False to skip the v1.0 repeat at the end


## Settings used in the paper

In [3]:
COUNTY_PREFIX = "42045"          # Delaware County block groups start with this
EJ_CUTOFF = 80                   # EJ Area = final score at or above the 80th percentile
DEPTHS_FT = [0.5, 1.0, 3.0]
DEPTH_TAGS = ["05", "10", "30"]  # used in column names, e.g. CFE_AREA_05
FLOOD_WEIGHT = 1.0               # w in Model 3
ROUNDING = 0.051                 # DEP publishes values rounded to 0.1

# The 32 PES indicators by component (field names in the DEP file)
EXPOSURES = ["OZONE", "PM25", "DIESELPM10", "TOXICAIREMISSIONS", "COMPRESSORSTATIONS",
             "TOXICWATEREMISSIONS", "PESTICIDE", "CHILDRENLEADRISK", "TRAFFIC"]
EFFECTS = ["UNCONVENTIONALOILGASWELL", "CONVENTIONALOILGASWELL", "RAILROADS", "LANDREMEDIATION",
           "HAZARDOUSWASTEANDSTORAGE", "COALMINING", "MUNICIPALWASTE", "IMPAIREDLAKESSTREAMS",
           "ABANDONEDMININGCONCERNS", "FLOODRISK"]
SENSITIVE = ["ASTHMA", "NOINSURANCE", "CANCER", "DISABILITY", "HEARTDISEASE"]
SOCIOECONOMIC = ["LOWEDUCATIONALATTAINMENT", "LINGUISTICISOLATION", "HOUSINGBURDEN", "POVERTY",
                 "UNEMPLOYMENT", "RACE", "OVER64", "UNDER5"]
ALL_INDICATORS = EXPOSURES + EFFECTS + SENSITIVE + SOCIOECONOMIC


def pct_cols(names):
    """DEP stores each indicator's percentile in a field ending in PCTILE."""
    return [name + "PCTILE" for name in names]


EXPOSURE_COLS = pct_cols(EXPOSURES)
EFFECT_COLS = pct_cols(EFFECTS)
EFFECT_COLS_NO_FLOOD = [c for c in EFFECT_COLS if c != "FLOODRISKPCTILE"]
SENSITIVE_COLS = pct_cols(SENSITIVE)
SOCIO_COLS = pct_cols(SOCIOECONOMIC)


def as_number(series):
    return pd.to_numeric(series, errors="coerce")


## PennEnviroScreen scoring rules (PA DEP 2026 methodology)

In [22]:
def pes_percentile(values, absent=None, denominator="present"):
    """Turn raw values into percentiles the way PES does.

    Block groups where the thing being measured is absent get 0 and are left out of the ranking.
    Everything else is ranked: percentile = (number with a value <= this one) / (number ranked) x 100.
    Missing values stay missing. A few DEP indicators rank against all block groups rather than
    only those where the phenomenon is present, so `denominator` can be "present" or "all".
    """
    v = as_number(pd.Series(values)).astype(float)
    if absent is None:
        absent = v.eq(0)
    absent = pd.Series(absent, index=v.index).fillna(False).astype(bool)

    if denominator == "present":
        ranked = v.notna() & ~absent
    else:
        ranked = v.notna()

    result = pd.Series(np.nan, index=v.index)
    if ranked.sum() > 0:
        result[ranked] = v[ranked].rank(method="max") / ranked.sum() * 100
    result[absent] = 0.0
    return result


def score_percentile(scores):
    """Percentile of a score among the block groups that have one."""
    scores = pd.Series(scores, dtype=float)
    result = pd.Series(np.nan, index=scores.index)
    has_score = scores.notna()
    result[has_score] = scores[has_score].rank(method="max") / has_score.sum() * 100
    return result


def mean_exposures(pct):
    """The documented Exposures score: a plain average of the nine indicator percentiles."""
    return pct.mean(axis=1)


def weighted_exposures(weights):
    """Exposures as a weighted average (used only in one sensitivity test)."""
    w = pd.Series(weights).reindex(EXPOSURE_COLS).values

    def score(pct):
        return (pct * w).sum(axis=1, min_count=1) / (pct.notna() * w).sum(axis=1)

    return score


def add_designation(scores, cutoff=EJ_CUTOFF):
    scores["FINAL_PCTL"] = score_percentile(scores["FINAL"])
    scores["EJ"] = scores["FINAL_PCTL"] >= cutoff
    return scores


def build_pes(pct, effect_cols=EFFECT_COLS, exposure_score=mean_exposures, cutoff=EJ_CUTOFF):
    """Rebuild PES from indicator percentiles.

    Each component is the average of its indicators. Pollution Burden gives Effects half the
    weight of Exposures; Population Characteristics averages the two population components.
    Both are rescaled to 0-10 by their highest value in the group being ranked (the state for
    official PES, the county for Models 1-3) and multiplied to get the final score.
    """
    s = pd.DataFrame(index=pct.index)
    s["EXP"] = exposure_score(pct[EXPOSURE_COLS])
    s["EE"] = pct[effect_cols].mean(axis=1)
    s["SP"] = pct[SENSITIVE_COLS].mean(axis=1)
    s["SOC"] = pct[SOCIO_COLS].mean(axis=1)
    s["PB"] = (s["EXP"] + 0.5 * s["EE"]) / 1.5
    s["PC"] = (s["SP"] + s["SOC"]) / 2
    s["PB_S"] = s["PB"] / s["PB"].max() * 10
    s["PC_S"] = s["PC"] / s["PC"].max() * 10
    s["FINAL"] = s["PB_S"] * s["PC_S"]
    return add_designation(s, cutoff)


def build_flood_domain(base, cfe_pctl, weight=FLOOD_WEIGHT, multiply=False, cutoff=EJ_CUTOFF):
    """Model 3: final = [(PB_S + w x CFE_S) / (1 + w)] x PC_S.
    With multiply=True this is Model 3A instead: PB_S x PC_S x CFE_S."""
    s = base[["EXP", "EE", "SP", "SOC", "PB", "PC", "PB_S", "PC_S"]].copy()
    s["CFE"] = pd.Series(cfe_pctl).reindex(s.index)
    s["CFE_S"] = s["CFE"] / s["CFE"].max() * 10
    if multiply:
        s["FINAL"] = s["PB_S"] * s["PC_S"] * s["CFE_S"]
    else:
        s["FINAL"] = (s["PB_S"] + weight * s["CFE_S"]) / (1 + weight) * s["PC_S"]
    return add_designation(s, cutoff)


def yes_no(series):
    """DEP's yes/no fields as True/False."""
    text = pd.Series(series).astype(str).str.strip().str.lower()
    return text.isin(["yes", "y", "true", "1", "1.0"])


def within_rounding(mine, published):
    """Share of block groups (%) where my value matches DEP's within rounding."""
    both = pd.concat([pd.Series(mine, dtype=float), pd.Series(published, dtype=float)], axis=1).dropna()
    diff = both.iloc[:, 0] - both.iloc[:, 1]
    return {"n": len(both), "mean_error": diff.mean(), "mean_abs_error": diff.abs().mean(),
            "max_abs_error": diff.abs().max(), "pct_within": (diff.abs() <= ROUNDING).mean() * 100}


## Step 1: read the data

In [5]:
def read_pes(version):
    import geopandas as gpd

    gdf = gpd.read_file(PES_FILES[version])
    gdf["GEOID"] = gdf["GEOID"].astype(str)
    if len(gdf) != STATE_ROWS[version]:
        print(f"Warning: {len(gdf):,} block groups in the {version} file, expected {STATE_ROWS[version]:,}")

    county_gdf = gdf[gdf["GEOID"].str.startswith(COUNTY_PREFIX)].copy()
    state = pd.DataFrame(gdf.drop(columns="geometry")).set_index("GEOID")
    county = state.loc[county_gdf["GEOID"]]

    # v1.1 has a separate field for the 80th-percentile rule; v1.0 only has EJAREA
    official_field = "PERCENTILE80THEJ" if "PERCENTILE80THEJ" in state.columns else "EJAREA"
    return state, county, county_gdf, official_field


def read_population(index):
    """2020 Census total population by block group (the second row of the download holds labels)."""
    if not POPULATION_FILE.exists():
        print("Census population file not found; resident counts are skipped.")
        return None
    d = pd.read_csv(POPULATION_FILE, dtype=str, skiprows=[1])
    d["GEOID"] = d["GEO_ID"].str[-12:]
    return as_number(d.set_index("GEOID")["P1_001N"]).reindex(index)


## Step 2: rebuild PES and check it against DEP's published scores

In [6]:
def choose_indicator_inputs(state):
    """For each indicator, try to reproduce DEP's statewide percentiles from the raw values.

    If the raw values reproduce at least 99% of the published percentiles, the county
    re-ranking uses the raw values. Otherwise it re-ranks DEP's published percentiles, which keeps
    the same order. (In v1.1 this happens for Land Remediation and Hazardous Waste and Storage.)
    """
    choice = {}
    for ind in ALL_INDICATORS:
        published = as_number(state[ind + "PCTILE"])
        raw = as_number(state[ind]).where(published.notna())
        best = None
        for denominator in ["present", "all"]:
            fit = within_rounding(pes_percentile(raw, absent=published.eq(0), denominator=denominator), published)
            if best is None or fit["pct_within"] > best[1]:
                best = (denominator, fit["pct_within"])
        if best[1] >= 99:
            choice[ind] = ("raw", best[0])
        else:
            choice[ind] = ("published", "present")
    return choice


def check_reconstruction(state, county, official_field):
    """Rebuild every score statewide from DEP's indicator percentiles and compare (Section 3.2)."""
    rebuilt = build_pes(state[pct_cols(ALL_INDICATORS)].apply(as_number))
    pairs = {"Environmental Exposures": ("EXP", "EXPOSURESSCORE"),
             "Environmental Effects": ("EE", "ENVEFFECTSSCORE"),
             "Sensitive Populations": ("SP", "SENSITIVEPOPSCORE"),
             "Socioeconomic Populations": ("SOC", "SOCIOECONOMICSCORE"),
             "Pollution Burden": ("PB", "POLLUTIONBURDENSCORE"),
             "Population Characteristics": ("PC", "POPULATIONCHARACTERISTICSSCORE"),
             "Final score": ("FINAL", "FINALSCORE"),
             "Final-score percentile": ("FINAL_PCTL", "FINALSCOREPCTILE")}
    rows = []
    for label, (mine, theirs) in pairs.items():
        fit = within_rounding(rebuilt[mine], as_number(state[theirs]))
        rows.append({"score": label, **fit})
    table = pd.DataFrame(rows).set_index("score")

    official = yes_no(state[official_field])
    agree_state = (rebuilt["EJ"] == official).mean() * 100
    agree_county = (rebuilt.loc[county.index, "EJ"] == official.loc[county.index]).mean() * 100
    return table, agree_state, agree_county


def fit_exposure_weights(state):
    """Weights that best reproduce DEP's published Exposures scores from the nine indicators
    (least squares, negative weights set to 0, rescaled to sum to 1). Sensitivity test only."""
    X = state[EXPOSURE_COLS].apply(as_number)
    y = as_number(state["EXPOSURESSCORE"])
    ok = X.notna().all(axis=1) & y.notna()
    w, *_ = np.linalg.lstsq(X[ok].values, y[ok].values, rcond=None)
    w = pd.Series(w, index=EXPOSURE_COLS).clip(lower=0)
    return w / w.sum()


## Step 3: Community Flood Exposure

In [7]:
def read_flood_shares(cache_file):
    """Read the saved flood shares and give the columns their current names."""
    d = pd.read_csv(cache_file, dtype={"GEOID": str}).set_index("GEOID")
    d.columns = [c.replace("CFVNC_", "CFE_NOCOAST_").replace("CFV_", "CFE_").replace("LEGACY_", "OLD_")
                 for c in d.columns]
    return d


def measure_flood_shares(county_gdf, cache_file):
    """Share of each block group's land flooded to 0.5, 1.0 and 3.0 ft or more.

    All three grids are put on the 0.61 m Lower Delaware grid with nearest-neighbor resampling, so
    depths are not changed. Where grids overlap, the deeper value is kept. Open water is removed
    and the share is taken of land area. Also saved:
      CFE_NOCOAST_*  the same without FEMA's coastal grid (sensitivity test)
      OLD_*          the original, uncorrected method: Lower Delaware grids only, overlapping
                     riverine and coastal cells added together, open water kept, total area
    This takes a while; the result is saved to cache_file and read from there next time.
    """
    import geopandas as gpd
    import rasterio
    from rasterio.vrt import WarpedVRT
    from rasterio.enums import Resampling
    from rasterio.windows import Window, transform as window_transform
    from rasterio.transform import Affine
    from rasterio.features import geometry_mask

    warnings.filterwarnings("ignore", message=".*nodata.*")
    raw = {name: rasterio.open(path) for name, path in DEPTH_GRIDS.items()}
    ref = raw["riverine_lower_delaware"]
    crs = ref.crs
    cell_w, cell_h = ref.transform.a, -ref.transform.e
    x_ref, y_ref = ref.transform.c, ref.transform.f
    bgs = county_gdf.to_crs(crs)

    # One grid covering the county plus 100 m, lined up with the Lower Delaware cells
    minx, miny, maxx, maxy = bgs.total_bounds
    left = x_ref + np.floor((minx - 100 - x_ref) / cell_w) * cell_w
    top = y_ref - np.floor((y_ref - (maxy + 100)) / cell_h) * cell_h
    width = int(np.ceil((maxx + 100 - left) / cell_w))
    height = int(np.ceil((top - (miny - 100)) / cell_h))
    grid = Affine(cell_w, 0.0, left, 0.0, -cell_h, top)
    cell_area = cell_w * cell_h

    def on_grid(src):
        return WarpedVRT(src, crs=crs, transform=grid, width=width, height=height,
                         resampling=Resampling.nearest, src_nodata=src.nodata, nodata=src.nodata)

    grids = {name: on_grid(src) for name, src in raw.items()}
    riverine = [i for i, name in enumerate(grids) if name.startswith("riverine")]
    old_method = [grids["riverine_lower_delaware"], grids["coastal_lower_delaware"]]

    water = gpd.read_file(WATER_FILE).to_crs(crs)
    water_all = water.geometry.union_all() if hasattr(water.geometry, "union_all") else water.unary_union

    def mask(shapes, shape, transform):
        shapes = [g for g in shapes if g is not None and not g.is_empty]
        if not shapes:
            return np.zeros(shape, dtype=bool)
        return geometry_mask(shapes, out_shape=shape, transform=transform, invert=True)

    def read(src, window):
        a = np.ma.filled(src.read(1, window=window, masked=True).astype(float), np.nan)
        a[a <= -1e30] = np.nan          # FEMA's no-data value is about -3.4e38
        return a

    def deepest(stack):
        has = ~np.isnan(stack)
        return np.where(has.any(axis=0), np.where(has, stack, -np.inf).max(axis=0), np.nan)

    rows = []
    for i, (geoid, geom) in enumerate(zip(bgs["GEOID"], bgs.geometry), 1):
        water_part = water_all.intersection(geom)
        total_area, water_area = geom.area, water_part.area
        land_area = total_area - water_area

        b = geom.bounds
        c0, c1 = int(np.floor((b[0] - left) / cell_w)), int(np.ceil((b[2] - left) / cell_w))
        r0, r1 = int(np.floor((top - b[3]) / cell_h)), int(np.ceil((top - b[1]) / cell_h))
        window = Window(c0, r0, c1 - c0, r1 - r0)
        transform = window_transform(window, grid)
        shape = (r1 - r0, c1 - c0)
        inside = mask([geom], shape, transform)
        land = inside & ~mask([water_part], shape, transform)

        stack = np.stack([read(g, window) for g in grids.values()])
        depth = deepest(stack)
        depth_no_coast = deepest(stack[riverine])
        on_land = depth[land]
        on_land = on_land[~np.isnan(on_land)]
        on_land_nc = depth_no_coast[land]
        on_land_nc = on_land_nc[~np.isnan(on_land_nc)]
        old_values = []
        for g in old_method:
            a = read(g, window)[inside]
            old_values.append(a[~np.isnan(a)])

        row = {"GEOID": geoid, "BG_AREA_M2": total_area, "WATER_AREA_M2": water_area,
               "LAND_AREA_M2": land_area}
        for t, tag in zip(DEPTHS_FT, DEPTH_TAGS):
            row[f"CFE_AREA_{tag}"] = (on_land >= t).sum() * cell_area / land_area * 100 if land_area > 0 else np.nan
            row[f"CFE_NOCOAST_AREA_{tag}"] = (on_land_nc >= t).sum() * cell_area / land_area * 100 if land_area > 0 else np.nan
            row[f"OLD_AREA_{tag}"] = sum((a >= t).sum() for a in old_values) * cell_area / total_area * 100
        row["overlap_cells"] = int((inside & ((~np.isnan(stack)).sum(axis=0) > 1)).sum())
        row["flood_max_land"] = float(on_land.max()) if on_land.size else 0.0
        all_inside = depth[inside]
        row["flood_max_all"] = float(np.nanmax(all_inside)) if np.isfinite(all_inside).any() else 0.0
        rows.append(row)
        if i % 50 == 0:
            print(f"  {i} of {len(bgs)} block groups done")

    shares = pd.DataFrame(rows).set_index("GEOID")
    cache_file.parent.mkdir(parents=True, exist_ok=True)
    shares.to_csv(cache_file)
    print("Saved flood shares to", cache_file)
    return shares


def flood_scores(shares, index):
    """Turn flood shares into percentiles with the PES rules.

    Each depth's share becomes a percentile (no flooded land = 0), the three are averaged, and the
    average is ranked again the same way, the way PES builds its own composite indicators.
    Also builds the versions used in the sensitivity tests.
    """
    cfe = pd.DataFrame(index=shares.index)
    for tag in DEPTH_TAGS:
        cfe[f"CFE_AREA_{tag}"] = shares[f"CFE_AREA_{tag}"]
        cfe[f"CFE_{tag}_PCTL"] = pes_percentile(shares[f"CFE_AREA_{tag}"])
    cfe["CFE_PCTL"] = pes_percentile(cfe[[f"CFE_{t}_PCTL" for t in DEPTH_TAGS]].mean(axis=1))

    # Without FEMA's coastal grid
    if "CFE_NOCOAST_AREA_05" in shares.columns:
        cfe["CFE_NOCOAST_AREA_05"] = shares["CFE_NOCOAST_AREA_05"]
        nc = pd.concat([pes_percentile(shares[f"CFE_NOCOAST_AREA_{t}"]) for t in DEPTH_TAGS], axis=1)
        cfe["CFE_NOCOAST_PCTL"] = pes_percentile(nc.mean(axis=1))

    # Raw share, no percentiles (average share rescaled to 0-100)
    raw = shares[[f"CFE_AREA_{t}" for t in DEPTH_TAGS]].mean(axis=1)
    cfe["CFE_RAW"] = raw / raw.max() * 100

    # Uncorrected version: the default ranking, which gives tied unflooded block groups their
    # average rank (about the 24th percentile) instead of 0
    old = pd.concat([shares[f"OLD_AREA_{t}"].rank(pct=True) * 100 for t in DEPTH_TAGS], axis=1)
    cfe["OLD_AREA_05"] = shares["OLD_AREA_05"]
    cfe["OLD_PCTL"] = old.mean(axis=1).rank(pct=True) * 100
    return cfe.reindex(index)


def describe_flooding(shares, cfe):
    """Numbers reported in Sections 2.2 and 3.1."""
    s = shares.reindex(cfe.index)
    flooded = cfe["CFE_PCTL"] > 0
    print(f"Block groups with land flooded 0.5 ft or more: {flooded.sum()} of {len(cfe)} "
          f"({flooded.mean() * 100:.1f}%); without flooded land: {(cfe['CFE_PCTL'] == 0).sum()}")
    print(f"Share of land flooded >= 0.5 ft: median {cfe['CFE_AREA_05'].median():.2f}%, "
          f"75th percentile {cfe['CFE_AREA_05'].quantile(0.75):.1f}%, maximum {cfe['CFE_AREA_05'].max():.1f}%")
    print(f"Block groups where the grids overlap: {(s['overlap_cells'] > 0).sum()}")
    print(f"Deepest modeled water including river channels: {s['flood_max_all'].max():.1f} ft "
          f"(on land: {s['flood_max_land'].max():.1f} ft)")
    print(f"Block groups containing open water: {(s['WATER_AREA_M2'] > 0).sum()}")
    print(f"Largest share flooded, uncorrected -> corrected: {cfe['OLD_AREA_05'].max():.1f}% -> "
          f"{cfe['CFE_AREA_05'].max():.1f}%")
    print(f"Spearman, uncorrected vs corrected CFE: {cfe['OLD_PCTL'].corr(cfe['CFE_PCTL'], method='spearman'):.2f}")
    print(f"Uncorrected percentile given to unflooded block groups: "
          f"{cfe.loc[cfe['CFE_PCTL'] == 0, 'OLD_PCTL'].min():.0f}")
    if "CFE_NOCOAST_AREA_05" in cfe:
        print(f"Block groups with flooded land without the coastal grid: {(cfe['CFE_NOCOAST_AREA_05'] > 0).sum()}")
    if "420459800001" in cfe.index:
        print(f"Tinicum special-use block group (airport) CFE percentile: {cfe.loc['420459800001', 'CFE_PCTL']:.1f}")

    # The riverfront block groups that dropped most once river channels were removed
    ranks = pd.DataFrame({"old_rank": cfe["OLD_PCTL"].rank(ascending=False, method="min"),
                          "new_rank": cfe["CFE_PCTL"].rank(ascending=False, method="min")})
    print("Uncorrected top 10 and their corrected ranks:")
    print(ranks.sort_values("old_rank").head(10).to_string())


def get_flood_scores(version, county_gdf, county_index):
    cache = CFE_CACHE[version]
    if cache.exists():
        shares = read_flood_shares(cache)
    else:
        print("No saved flood shares; measuring them from the depth grids (slow).")
        shares = measure_flood_shares(county_gdf, cache)
    return shares, flood_scores(shares, county_index)


## Step 4: the models

In [8]:
def county_percentiles(county, choice):
    """Re-rank all 32 indicators within Delaware County using the PES rules (Model 1 inputs).
    Block groups DEP left blank stay blank, and DEP's zeros stay zero."""
    pct = pd.DataFrame(index=county.index)
    for ind in ALL_INDICATORS:
        published = as_number(county[ind + "PCTILE"])
        source, denominator = choice[ind]
        values = as_number(county[ind]) if source == "raw" else published
        pct[ind + "PCTILE"] = pes_percentile(values.where(published.notna()),
                                             absent=published.eq(0), denominator=denominator)
    return pct


def official_pes(county, official_field):
    """Model 0: DEP's published scores and official designations."""
    return pd.DataFrame({"PB": as_number(county["POLLUTIONBURDENSCORE"]),
                         "PC": as_number(county["POPULATIONCHARACTERISTICSSCORE"]),
                         "FINAL": as_number(county["FINALSCORE"]),
                         "FINAL_PCTL": as_number(county["FINALSCOREPCTILE"]),
                         "EJ": yes_no(county[official_field]).values}, index=county.index)


def build_models(county, county_pct, cfe, official_field):
    m0 = official_pes(county, official_field)
    m1 = build_pes(county_pct)

    with_cfe = county_pct.copy()
    with_cfe["FLOODRISKPCTILE"] = cfe["CFE_PCTL"]
    m2 = build_pes(with_cfe)

    # Model 3 drops Flood Risk from Effects so flooding is not counted twice
    m3_base = build_pes(county_pct, effect_cols=EFFECT_COLS_NO_FLOOD)
    m3 = build_flood_domain(m3_base, cfe["CFE_PCTL"])
    m3a = build_flood_domain(m1, cfe["CFE_PCTL"], multiply=True)
    return {"M0": m0, "M1": m1, "M2": m2, "M3": m3, "M3A": m3a}, m3_base


## Step 5: who gains and loses designation

In [9]:
def compare(a, b):
    """Retained, gained and lost EJ Areas between two models, with overlap and rank correlation."""
    A = a["EJ"].astype(bool)
    B = b["EJ"].reindex(A.index).fillna(False).astype(bool)
    either = (A | B).sum()
    return {"from": int(A.sum()), "to": int(B.sum()),
            "retained": int((A & B).sum()), "gained": int((~A & B).sum()), "lost": int((A & ~B).sum()),
            "jaccard": (A & B).sum() / either if either else np.nan,
            "spearman": a["FINAL"].corr(b["FINAL"].reindex(A.index), method="spearman")}


def change_type(a, b):
    A, B = a["EJ"].astype(bool), b["EJ"].reindex(a.index).fillna(False).astype(bool)
    out = pd.Series("Not EJ", index=A.index)
    out[A & B] = "Retained"
    out[~A & B] = "Gained"
    out[A & ~B] = "Lost"
    return out


def profile(models, a, b, cfe, pop):
    """Describe the block groups that were retained, gained, lost or never designated (Table 3).
    PB and PC percentiles are county percentiles from Model 1."""
    m1 = models["M1"]
    d = pd.DataFrame({"change": change_type(models[a], models[b]),
                      "cfe": cfe["CFE_PCTL"],
                      "no_flood": cfe["CFE_PCTL"].eq(0),
                      "pc_pct": score_percentile(m1["PC"]),
                      "pb_pct": score_percentile(m1["PB"])})
    d["pc_below_median"] = d["pc_pct"] < 50
    d["people"] = pop.reindex(d.index) if pop is not None else np.nan
    g = d.groupby("change").agg(n=("change", "size"), mean_cfe=("cfe", "mean"),
                                pct_no_flood=("no_flood", "mean"), median_pc=("pc_pct", "median"),
                                median_pb=("pb_pct", "median"), pc_below_median=("pc_below_median", "sum"),
                                residents=("people", "sum"))
    g["pct_no_flood"] *= 100
    return g.reindex(["Retained", "Gained", "Lost", "Not EJ"])


def print_comparison(label, c):
    print(f"{label}: retained {c['retained']}, gained {c['gained']}, lost {c['lost']} "
          f"(Jaccard {c['jaccard']:.2f}, Spearman {c['spearman']:.3f})")


## Step 6: sensitivity tests (Tables 4 and 5)

In [10]:
def cutoff_tests(models, cutoffs=(75, 80, 85, 90)):
    """Repeat the comparisons with other designation cutoffs."""
    print("\nCutoff   official/county   M0->M1 gained/lost (J)   M1->M2 gained/lost (J)   M1->M3 gained/lost (J)")
    for cut in cutoffs:
        m = {k: add_designation(models[k].drop(columns=["FINAL_PCTL", "EJ"]).copy(), cut) for k in ("M1", "M2", "M3")}
        m["M0"] = models["M0"].copy()
        m["M0"]["EJ"] = models["M0"]["FINAL_PCTL"] >= cut
        parts = []
        for a, b in [("M0", "M1"), ("M1", "M2"), ("M1", "M3")]:
            c = compare(m[a], m[b])
            parts.append(f"{c['gained']} / {c['lost']} ({c['jaccard']:.2f})")
        c01 = compare(m["M0"], m["M1"])
        print(f"{cut}th      {c01['from']} / {c01['to']}          " + "        ".join(parts))


def specification_tests(models, county_pct, m3_base, cfe, exposure_weights):
    """Table 5: change one choice at a time and count swaps relative to Model 1."""
    m1 = models["M1"]
    n_ej = int(m1["EJ"].sum())

    def model2_with(flood_pctl, pct=county_pct, **kw):
        p = pct.copy()
        p["FLOODRISKPCTILE"] = flood_pctl
        return build_pes(p, **kw)

    def report(name, m2=None, m3=None, own_m1=None):
        ref = m1 if own_m1 is None else own_m1
        text = f"{name:48s}"
        text += f"M1->M2 {compare(ref, m2)['gained']:>2} / {compare(ref, m2)['lost']:<2}   " if m2 is not None else " " * 17
        if m3 is not None:
            c = compare(ref, m3)
            text += f"M1->M3 {c['gained']} / {c['lost']} (Jaccard {c['jaccard']:.2f})"
        if own_m1 is not None:
            text += f"   [Model 1 itself changes {compare(m1, own_m1)['gained']} of {n_ej}]"
        print(text)

    print()
    versions = [("Main specification", cfe["CFE_PCTL"]),
                ("Depth >= 0.5 ft only", cfe["CFE_05_PCTL"]),
                ("Depth >= 1.0 ft only", cfe["CFE_10_PCTL"]),
                ("Depth >= 3.0 ft only", cfe["CFE_30_PCTL"]),
                ("Raw flooded-land share (no percentiles)", cfe["CFE_RAW"]),
                ("Uncorrected CFE", cfe["OLD_PCTL"])]
    for name, flood in versions:
        report(name, m2=model2_with(flood), m3=build_flood_domain(m3_base, flood))

    for w in (0.5, 2.0):
        report(f"Model 3 weight w = {w:g}", m3=build_flood_domain(m3_base, cfe["CFE_PCTL"], weight=w))
    report("Flood Risk kept in Environmental Effects", m3=build_flood_domain(m1, cfe["CFE_PCTL"]))

    if "CFE_NOCOAST_PCTL" in cfe:
        no_coast = cfe["CFE_NOCOAST_PCTL"]
        m3_no_coast = build_flood_domain(m3_base, no_coast)
        report("Coastal depth grid left out", m2=model2_with(no_coast), m3=m3_no_coast)
        both = (models["M3"]["EJ"] & m3_no_coast["EJ"]).sum()
        print(f"{'':48s}Model 3 with and without the coastal grid share {both} of {n_ej} EJ Areas")

    # Missing socioeconomic indicators counted as 0 instead of skipped
    # (block groups with no socioeconomic data at all stay unscored)
    has_socio = county_pct[SOCIO_COLS].notna().any(axis=1)
    zeros = county_pct.copy()
    zeros.loc[has_socio, SOCIO_COLS] = zeros.loc[has_socio, SOCIO_COLS].fillna(0)
    report("Missing socioeconomic indicators set to zero",
           m2=model2_with(cfe["CFE_PCTL"], pct=zeros),
           m3=build_flood_domain(build_pes(zeros, effect_cols=EFFECT_COLS_NO_FLOOD), cfe["CFE_PCTL"]),
           own_m1=build_pes(zeros))

    # Exposures rebuilt with weights fitted to DEP's published Exposures scores
    fitted = weighted_exposures(exposure_weights)
    report("Exposures weights fitted to published scores",
           m2=model2_with(cfe["CFE_PCTL"], exposure_score=fitted),
           m3=build_flood_domain(build_pes(county_pct, effect_cols=EFFECT_COLS_NO_FLOOD, exposure_score=fitted),
                                 cfe["CFE_PCTL"]),
           own_m1=build_pes(county_pct, exposure_score=fitted))


## Run everything

In [11]:
def run_version(version):
    """Read one PES release, measure flooding, and build the models."""
    state, county, county_gdf, official_field = read_pes(version)
    choice = choose_indicator_inputs(state)
    shares, cfe = get_flood_scores(version, county_gdf, county.index)
    county_pct = county_percentiles(county, choice)
    models, m3_base = build_models(county, county_pct, cfe, official_field)
    return dict(state=state, county=county, official_field=official_field, choice=choice,
                shares=shares, cfe=cfe, county_pct=county_pct, models=models, m3_base=m3_base)


In [12]:
pd.set_option("display.width", 160)

print("=== PennEnviroScreen v1.1, Delaware County ===")
r = run_version("v1.1")
state, county, cfe, models = r["state"], r["county"], r["cfe"], r["models"]
pop = read_population(county.index)


=== PennEnviroScreen v1.1, Delaware County ===


### Section 2.1: the data

In [13]:
n_complete = county[pct_cols(ALL_INDICATORS)].apply(as_number).notna().all(axis=1).sum()
n_scored = as_number(county["FINALSCORE"]).notna().sum()
print(f"Block groups: {len(state):,} statewide, {len(county)} in Delaware County "
      f"({n_complete} with all 32 indicators, {n_scored} scored)")
if pop is not None:
    print(f"Delaware County residents (2020 Census): {pop.sum():,.0f}")
if "PERCENTILE80THEJ" in county.columns:
    # The final EJ Policy also designates unscored block groups in the top 5% of Pollution Burden
    pb_only = yes_no(county["EJAREA"]) & ~yes_no(county["PERCENTILE80THEJ"])
    print(f"EJ Areas added in Delaware County by the top-5% Pollution Burden rule: {pb_only.sum()}")


Block groups: 10,172 statewide, 417 in Delaware County (400 with all 32 indicators, 415 scored)
Delaware County residents (2020 Census): 576,830
EJ Areas added in Delaware County by the top-5% Pollution Burden rule: 0


### Section 3.2: can PES be rebuilt from its documentation?

In [14]:
print("\n--- Rebuilding PES statewide (Section 3.2)")
reproduced = [k for k, (src, _) in r["choice"].items() if src == "raw"]
print(f"Indicators reproduced from raw values: {len(reproduced)} of 32; "
      f"not reproduced: {[k for k in ALL_INDICATORS if k not in reproduced]}")
check, agree_state, agree_county = check_reconstruction(state, county, r["official_field"])
print(check.round(3).to_string())
print(f"Designations matching the official EJ Areas: {agree_state:.1f}% statewide, "
      f"{agree_county:.1f}% in Delaware County")



--- Rebuilding PES statewide (Section 3.2)
Indicators reproduced from raw values: 30 of 32; not reproduced: ['LANDREMEDIATION', 'HAZARDOUSWASTEANDSTORAGE']
                                n  mean_error  mean_abs_error  max_abs_error  pct_within
score                                                                                   
Environmental Exposures     10172      -3.404           4.055         11.700       0.374
Environmental Effects       10172      -0.000           0.025          0.050     100.000
Sensitive Populations       10110       0.001           0.024          0.050     100.000
Socioeconomic Populations   10115       0.000           0.025          0.050     100.000
Pollution Burden            10172      -2.269           2.703          7.770       0.551
Population Characteristics  10110       0.001           0.033          0.095      75.321
Final score                 10110      -0.326           1.354          8.678       2.671
Final-score percentile      10110      -0.

### Section 3.1: flood exposure

In [15]:
print("\n--- Flood exposure (Section 3.1)")
describe_flooding(r["shares"], cfe)



--- Flood exposure (Section 3.1)
Block groups with land flooded 0.5 ft or more: 221 of 417 (53.0%); without flooded land: 196
Share of land flooded >= 0.5 ft: median 0.22%, 75th percentile 6.2%, maximum 67.3%
Block groups where the grids overlap: 11
Deepest modeled water including river channels: 63.6 ft (on land: 54.8 ft)
Block groups containing open water: 60
Largest share flooded, uncorrected -> corrected: 80.9% -> 67.3%
Spearman, uncorrected vs corrected CFE: 0.99
Uncorrected percentile given to unflooded block groups: 24
Block groups with flooded land without the coastal grid: 209
Tinicum special-use block group (airport) CFE percentile: 96.8
Uncorrected top 10 and their corrected ranks:
              old_rank  new_rank
GEOID                           
420454037012       1.0       1.0
420454034022       2.0       2.0
420454107001       3.0      47.0
420454037022       4.0      18.0
420459800001       5.0       8.0
420454065002       6.0      17.0
420454027002       7.0       3.0


### Models

In [16]:
print("\n--- Models")
for name, m in models.items():
    print(f"{name}: {int(m['EJ'].sum())} EJ Areas")
print(f"Model 3A block groups with a final score of 0: {int(models['M3A']['FINAL'].eq(0).sum())} of {n_scored}")



--- Models
M0: 104 EJ Areas
M1: 84 EJ Areas
M2: 84 EJ Areas
M3: 84 EJ Areas
M3A: 84 EJ Areas
Model 3A block groups with a final score of 0: 195 of 415


### Section 3.3: state versus county ranking

In [17]:
print("\n--- Statewide versus county ranking (Section 3.3)")
c = compare(models["M0"], models["M1"])
print_comparison("Model 0 -> Model 1", c)
if pop is not None:
    for k in ("M0", "M1"):
        print(f"Residents of {k} EJ Areas: {pop[models[k]['EJ'].astype(bool)].sum():,.0f}")
shift = (r["county_pct"] - county[r["county_pct"].columns].apply(as_number)).mean()
shift.index = shift.index.str.replace("PCTILE", "")
print("Average change in indicator percentile, county minus statewide:")
print(shift.sort_values().round(1).to_string())
p01 = profile(models, "M0", "M1", cfe, pop)
print(p01[["n", "median_pb", "median_pc"]].round(1).to_string())



--- Statewide versus county ranking (Section 3.3)
Model 0 -> Model 1: retained 83, gained 1, lost 21 (Jaccard 0.79, Spearman 0.969)
Residents of M0 EJ Areas: 142,306
Residents of M1 EJ Areas: 108,855
Average change in indicator percentile, county minus statewide:
OZONE                      -39.9
DIESELPM10                 -25.3
PM25                       -23.1
IMPAIREDLAKESSTREAMS       -15.1
CHILDRENLEADRISK           -13.9
RACE                       -13.1
HOUSINGBURDEN              -10.1
LANDREMEDIATION             -6.6
UNEMPLOYMENT                -4.6
UNDER5                      -4.2
RAILROADS                   -3.6
MUNICIPALWASTE              -2.5
TOXICAIREMISSIONS           -1.3
FLOODRISK                   -0.6
LINGUISTICISOLATION         -0.5
TRAFFIC                     -0.4
TOXICWATEREMISSIONS         -0.4
CONVENTIONALOILGASWELL       0.0
UNCONVENTIONALOILGASWELL     0.0
COALMINING                   0.0
ABANDONEDMININGCONCERNS      0.0
COMPRESSORSTATIONS           0.2
ASTHMA   

### Section 3.4: CFE as a single indicator

In [18]:
print("\n--- CFE replacing Flood Risk (Section 3.4)")
print_comparison("Model 1 -> Model 2", compare(models["M1"], models["M2"]))
rho = r["county_pct"]["FLOODRISKPCTILE"].corr(cfe["CFE_PCTL"], method="spearman")
effects_change = models["M2"]["EE"] - models["M1"]["EE"]
print(f"Spearman, PES Flood Risk vs CFE: {rho:.2f}")
print(f"Change in Environmental Effects score: mean {effects_change.mean():.1f}, "
      f"range {effects_change.min():.1f} to {effects_change.max():.1f}")



--- CFE replacing Flood Risk (Section 3.4)
Model 1 -> Model 2: retained 84, gained 0, lost 0 (Jaccard 1.00, Spearman 0.999)
Spearman, PES Flood Risk vs CFE: 0.72
Change in Environmental Effects score: mean -0.9, range -9.4 to 5.4


### Section 3.5: CFE as its own domain

In [19]:
print("\n--- CFE as a separate domain (Section 3.5)")
print_comparison("Model 1 -> Model 3", compare(models["M1"], models["M3"]))
print_comparison("Model 1 -> Model 3A", compare(models["M1"], models["M3A"]))
print_comparison("Model 0 -> Model 3", compare(models["M0"], models["M3"]))
print("Model 1 -> Model 3, by change in status:")
print(profile(models, "M1", "M3", cfe, pop).round(1).to_string())
print("Model 1 -> Model 3A, by change in status:")
print(profile(models, "M1", "M3A", cfe, pop)[["n", "pc_below_median"]].to_string())
official_no_flood = (models["M0"]["EJ"].astype(bool) & cfe["CFE_PCTL"].eq(0)).sum()
print(f"Official EJ Areas with no land flooded 0.5 ft or more: {official_no_flood} of {int(models['M0']['EJ'].sum())}")



--- CFE as a separate domain (Section 3.5)
Model 1 -> Model 3: retained 50, gained 34, lost 34 (Jaccard 0.42, Spearman 0.859)
Model 1 -> Model 3A: retained 45, gained 39, lost 39 (Jaccard 0.37, Spearman 0.379)
Model 0 -> Model 3: retained 60, gained 24, lost 44 (Jaccard 0.47, Spearman 0.843)
Model 1 -> Model 3, by change in status:
            n  mean_cfe  pct_no_flood  median_pc  median_pb  pc_below_median  residents
change                                                                                 
Retained   50      69.7           6.0       90.7       92.8                0      62111
Gained     34      76.8           0.0       60.6       63.2                8      49301
Lost       34       4.7          73.5       85.4       85.4                0      46744
Not EJ    299      16.2          56.2       35.9       36.2              199     418674
Model 1 -> Model 3A, by change in status:
            n  pc_below_median
change                        
Retained   45                0
Ga

### Section 3.6: sensitivity tests

In [20]:
print("\n--- Sensitivity tests (Section 3.6)")
cutoff_tests(models)
specification_tests(models, r["county_pct"], r["m3_base"], cfe, fit_exposure_weights(state))



--- Sensitivity tests (Section 3.6)

Cutoff   official/county   M0->M1 gained/lost (J)   M1->M2 gained/lost (J)   M1->M3 gained/lost (J)
75th      118 / 104          4 / 18 (0.82)        4 / 4 (0.93)        34 / 34 (0.51)
80th      104 / 84          1 / 21 (0.79)        0 / 0 (1.00)        34 / 34 (0.42)
85th      85 / 63          0 / 22 (0.74)        1 / 1 (0.97)        24 / 24 (0.45)
90th      63 / 42          0 / 21 (0.67)        1 / 1 (0.95)        18 / 18 (0.40)

Main specification                              M1->M2  0 / 0    M1->M3 34 / 34 (Jaccard 0.42)
Depth >= 0.5 ft only                            M1->M2  0 / 0    M1->M3 33 / 33 (Jaccard 0.44)
Depth >= 1.0 ft only                            M1->M2  0 / 0    M1->M3 34 / 34 (Jaccard 0.42)
Depth >= 3.0 ft only                            M1->M2  1 / 1    M1->M3 36 / 36 (Jaccard 0.40)
Raw flooded-land share (no percentiles)         M1->M2  1 / 1    M1->M3 14 / 14 (Jaccard 0.71)
Uncorrected CFE                                 M1-

### PES v1.0 on 2010 block groups

In [21]:
if RUN_VERSION_1_0 and PES_FILES["v1.0"].exists():
    print("\n=== PennEnviroScreen v1.0 (2010 block groups) ===")
    r10 = run_version("v1.0")
    m10 = r10["models"]
    print(f"Delaware County block groups: {len(r10['county'])}; "
          f"EJ Areas: official {int(m10['M0']['EJ'].sum())}, county {int(m10['M1']['EJ'].sum())}")
    print_comparison("Model 0 -> Model 1", compare(m10["M0"], m10["M1"]))
    print_comparison("Model 1 -> Model 2", compare(m10["M1"], m10["M2"]))
    print_comparison("Model 1 -> Model 3", compare(m10["M1"], m10["M3"]))



=== PennEnviroScreen v1.0 (2010 block groups) ===
Delaware County block groups: 437; EJ Areas: official 88, county 88
Model 0 -> Model 1: retained 79, gained 9, lost 9 (Jaccard 0.81, Spearman 0.964)
Model 1 -> Model 2: retained 86, gained 2, lost 2 (Jaccard 0.96, Spearman 0.998)
Model 1 -> Model 3: retained 53, gained 35, lost 35 (Jaccard 0.43, Spearman 0.833)
